# Query-Conditioned Recommendation Dataset Construction

This notebook constructs a synthetic query-conditioned recommendation dataset from the Amazon Reviews 2023 Electronics dataset.

The workflow consists of:

1. Load review and metadata datasets.
2. Explore the data.
3. Filter active users.
4. Construct chronological user histories.
5. Generate ground-truth interactions.
6. Join product metadata.
7. Save the processed dataset for downstream recommendation experiments.

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path

pd.set_option("display.max_columns", None)
tqdm.pandas()

In [2]:
PROJECT_DIR= Path.home()/"query-conditioned-recommender"
RAW_DATA= PROJECT_DIR/"data"/"raw"
PROCESSED_DATA= PROJECT_DIR/"data"/"processed"
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)
REVIEWS_PATH= RAW_DATA/"Electronics.jsonl"
META_PATH= RAW_DATA/"meta_Electronics.jsonl"

print(REVIEWS_PATH)
print(META_PATH)

/users/adhp511/query-conditioned-recommender/data/raw/Electronics.jsonl
/users/adhp511/query-conditioned-recommender/data/raw/meta_Electronics.jsonl


## Load a Sample

The complete Amazon Reviews dataset is very large.
A small sample is first loaded to inspect the data structure before processing the full dataset.

In [3]:
sample_reviews= pd.read_json(
    REVIEWS_PATH,
    lines=True,
    nrows=10000
)
sample_meta= pd.read_json(
    META_PATH,
    lines=True,
    nrows=5000
)

In [4]:
print(sample_reviews.shape)
print(sample_meta.shape)

(10000, 10)
(5000, 16)


In [5]:
sample_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,3,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,[{'small_image_url': 'https://m.media-amazon.c...,B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2022-07-18 22:58:37.948,0,True
1,1,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,[],B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2020-06-20 18:42:29.731,0,True
2,5,Excellent!,I love these. They even come with a carry case...,[],B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2018-04-07 09:23:37.534,0,True
3,5,Great laptop backpack!,I was searching for a sturdy backpack for scho...,[],B001OC5JKY,B001OC5JKY,AGGZ357AO26RQZVRLGU4D4N52DZQ,2010-11-20 18:41:35.000,18,True
4,5,Best Headphones in the Fifties price range!,I've bought these headphones three times becau...,[],B013J7WUGC,B07CJYMRWM,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,2023-02-17 02:39:41.238,0,True


In [6]:
sample_meta.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,All Electronics,FS-1051 FATSHARK TELEPORTER V3 HEADSET,3.5,6,[],[Teleporter V3 The “Teleporter V3” kit sets a ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Fat Shark,"[Electronics, Television & Video, Video Glasses]","{'Date First Available': 'August 2, 2014', 'Ma...",B00MCW7G9M,NaN,NaN,NaN
1,All Electronics,Ce-H22B12-S1 4Kx2K Hdmi 4Port,5.0,1,"[UPC: 662774021904, Weight: 0.600 lbs]",[HDMI In - HDMI Out],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],SIIG,"[Electronics, Television & Video, Accessories,...",{'Product Dimensions': '0.83 x 4.17 x 2.05 inc...,B00YT6XQSE,NaN,NaN,NaN
2,Computers,Digi-Tatoo Decal Skin Compatible With MacBook ...,4.5,246,[WARNING: Please IDENTIFY MODEL NUMBER on the ...,[],19.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'AL 2Sides Video', 'url': 'https://...",Digi-Tatoo,"[Electronics, Computers & Accessories, Laptop ...","{'Brand': 'Digi-Tatoo', 'Color': 'Fresh Marble...",B07SM135LS,NaN,NaN,NaN
3,AMAZON FASHION,NotoCity Compatible with Vivoactive 4 band 22m...,4.5,233,[☛NotoCity 22mm band is designed for Vivoactiv...,[],9.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],NotoCity,"[Electronics, Wearable Technology, Clips, Arm ...","{'Date First Available': 'May 29, 2020', 'Manu...",B089CNGZCW,NaN,NaN,NaN
4,Cell Phones & Accessories,Motorola Droid X Essentials Combo Pack,3.8,64,"[New Droid X Essentials Combo Pack, Exclusive ...",[all Genuine High Quality Motorola Made Access...,14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Verizon,"[Electronics, Computers & Accessories, Compute...",{'Product Dimensions': '11.6 x 6.9 x 3.1 inche...,B004E2Z88O,NaN,NaN,NaN


In [7]:
print(sample_reviews.columns.tolist())
print(sample_meta.columns.tolist())

['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']


## Keep Relevant Columns

Only the columns required for dataset construction are retained.

In [8]:
sample_reviews= sample_reviews[
    [
        "user_id",
        "parent_asin",
        "timestamp",
        "rating",
        "verified_purchase",
        "title",
        "text"
    ]
]
sample_meta= sample_meta[
    [
        "parent_asin",
        "title",
        "categories",
        "description"
    ]
]

In [9]:
sample_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   user_id            10000 non-null  str           
 1   parent_asin        10000 non-null  str           
 2   timestamp          10000 non-null  datetime64[ms]
 3   rating             10000 non-null  int64         
 4   verified_purchase  10000 non-null  bool          
 5   title              10000 non-null  str           
 6   text               10000 non-null  str           
dtypes: bool(1), datetime64[ms](1), int64(1), str(4)
memory usage: 4.9 MB


In [10]:
sample_meta.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   parent_asin  5000 non-null   str   
 1   title        5000 non-null   str   
 2   categories   5000 non-null   object
 3   description  5000 non-null   object
dtypes: object(2), str(2)
memory usage: 824.0+ KB


## Filter Active Users

Only users with at least six interactions are retained.
This allows five historical interactions and one future interaction that will serve as the ground-truth recommendation.

In [11]:
user_counts= sample_reviews["user_id"].value_counts()
active_users= user_counts[user_counts >= 6].index
filtered_reviews= sample_reviews[
    sample_reviews["user_id"].isin(active_users)
].copy()
print(filtered_reviews.shape)

(7509, 7)


In [12]:
filtered_reviews.head()

,user_id,parent_asin,timestamp,rating,verified_purchase,title,text
5,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B07CML419K,2021-11-21 19:28:01.041,5,True,Great Fan! I’m a FAN!,"Light weight, quiet and totally awesome!!! It ..."
6,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B07BHHB5RH,2019-08-06 22:34:39.386,5,True,solid sound for the price,Update 2-they sent a new warranty replacement....
7,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B09S6Y5BRG,2018-11-04 18:40:31.659,5,True,Love the headphones - great range,These are fantastic headphones and I love that...
8,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B01LW71IBJ,2016-02-29 19:02:51.000,5,True,Five Stars,pretty good for the price.
9,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,B017T99JPG,2016-02-29 18:59:25.000,5,True,BUY THIS THANG,yes.. so good. just buy it. my favorite featu...


## Sort User Interactions

Interactions are ordered chronologically to preserve the sequence of user behaviour.

In [13]:
filtered_reviews= filtered_reviews.sort_values(["user_id", "timestamp"]).reset_index(drop=True)
filtered_reviews.head(20)

,user_id,parent_asin,timestamp,rating,verified_purchase,title,text
0,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B075B5T7L5,2015-09-01 12:14:16.000,5,True,I can assure that this micro sd card installed...,After reading so many other comments online to...
1,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B00WA4PSEM,2015-09-01 12:23:56.000,5,True,Big sound from a tiny Bluetooth speaker.,Paired up easily with my phone and has excelle...
2,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B09VS4V18K,2016-10-02 21:27:51.000,5,True,"I've never had a WD drive die on me yet, and I...",This WD 1 TB Performance Desktop Hard Disk get...
3,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B00G2UTVH0,2016-10-02 22:15:43.000,5,True,These fit my 5 Panasonic Handsets when the bat...,These fit my 5 Panasonic Handsets and worked p...
4,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B07RS68PJ7,2016-10-23 09:32:31.000,5,True,Not just battery backup but affordable voltage...,I needed battery backup and AVR because of ele...
5,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B007VKII6A,2017-05-21 14:36:52.000,2,True,"IDance Headphones came in wrong color, 2x colu...",First my wife was excited I ordered her the WH...
6,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B07454F4JH,2017-08-24 17:49:29.844,5,True,Second of these that I bought because of relia...,Second of these that I bought because of relia...
7,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B00SMVLPIK,2017-08-24 17:57:36.638,5,True,Got this for a friend to use these 3.5mm corde...,Got this for a friend to use these 3.5mm corde...
8,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B0179DDIQI,2017-08-24 18:06:02.421,5,True,These are a great sounding headset and afforda...,These are a great sounding headset and very af...
9,AE2MIK6YBTBRNZZGI5O5Z6VCMEPA,B00X088DZI,2019-05-11 17:22:30.699,5,True,These look like and perform even better than t...,I have a Panasonic KXTG6700B 2-line Phone/Ans...


In [14]:
filtered_reviews.to_parquet(PROCESSED_DATA/"filtered_reviews.parquet",index=False)